# Chương 2: MLP bằng PyTorch (`nn.Sequential`, autograd)
Cùng kiến trúc, khởi tạo He, SGD thuần, tốc độ học, epoch, kích thước lô với hai cài đặt còn lại (xem `CFG` trong `ch2_run.py`). 
Notebook chạy nhanh một bộ dữ liệu một hạt giống; muốn chạy đủ 3 bộ x 3 hạt giống dùng `python ch2_run.py torch`.

In [1]:
import os, sys, inspect
sys.path.insert(0, os.path.abspath(".."))
os.chdir(os.path.abspath(".."))
import numpy as np
import ch2_run as C

In [2]:
print(inspect.getsource(C.run_torch)) if "torch" != "numpy" else print(inspect.getsource(__import__("dl_scratch").MLPScratch))

def run_torch(name, D, c, seed):
    import torch, torch.nn as nn
    torch.set_num_threads(4); torch.manual_seed(seed)
    layers = []
    for i, (a, b) in enumerate(zip(c["sizes"][:-1], c["sizes"][1:])):
        lin = nn.Linear(a, b); nn.init.kaiming_normal_(lin.weight, nonlinearity="relu"); nn.init.zeros_(lin.bias)
        layers.append(lin)
        if i < len(c["sizes"]) - 2: layers.append(nn.ReLU())
    net = nn.Sequential(*layers); opt = torch.optim.SGD(net.parameters(), lr=c["lr"])
    X = torch.tensor(D["X_tr"], dtype=torch.float32)
    y = torch.tensor(D["y_tr"], dtype=torch.long if c["task"] == "multiclass" else torch.float32)
    if c["task"] == "multiclass": y = y.view(-1)
    lossf = {"binary": nn.BCEWithLogitsLoss(), "regression": lambda o, t: 0.5 * nn.functional.mse_loss(o, t), "multiclass": nn.CrossEntropyLoss()}[c["task"]]
    g = torch.Generator().manual_seed(seed); N = len(X); bs = c["batch"] or N; hist = []
    t0 = time.perf_counter()
    for ep in range(c["epochs"

In [3]:
name = "housing"                      # đổi thành "diabetes" hoặc "comments" (diabetes chạy lâu hơn)
D = dict(np.load("data/ch2_%s.npz" % name))
metrics, seconds, n_params, loss = C.run_torch(name, D, C.CFG[name], 42)
print(n_params, "tham số;", round(seconds, 1), "giây")
print({k: round(v, 4) for k, v in metrics.items()})

10369 tham số; 6.4 giây
{'r2_log': 0.649, 'r2_price': 0.5867, 'mae': 1.637, 'rmse': 3.9288, 'mape': 0.2987}
